In [1]:
# ============================================================
# CELDA 1: CONFIGURACIÓN Y SETUP
# ============================================================

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
import calendar

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_rows", 100)

# ============================================================
# RUTAS
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\Jaime Valderrama\OneDrive - American Sportswear, S.A\Documentos\Jaime")
BI_PATH = PROJECT_ROOT / "Ventas (Total Año)" / "BI"

FOLDER_INPUT = BI_PATH

# 🔧 CORRECCIÓN: ahora se crea en "Ventas (Total Año)"
FOLDER_OUTPUT = BI_PATH.parent / "Reportes_Consolidados"
FOLDER_OUTPUT.mkdir(parents=True, exist_ok=True)

DATA_PROCESSED = PROJECT_ROOT / "miprimerproyectods" / "data" / "processed"
MAESTROS_PATH = BI_PATH / "Maestros"

print("📂 Rutas configuradas\n")
print(f"Origen ventas: {FOLDER_INPUT}")
print(f"Salida reportes: {FOLDER_OUTPUT}")
print(f"Processed: {DATA_PROCESSED}")
print(f"Maestros: {MAESTROS_PATH}")

# ============================================================
# PARÁMETROS DEL REPORTE
# ============================================================

CY = 2026
LY = 2025
MESES = [4]   # Abril. Ejemplo: [1,2,3,4] = acumulado ene-abr

OUTPUT_FILENAME = f"Ventas_Consolidadox_{CY}_vs_{LY}_meses_{'-'.join(str(m).zfill(2) for m in MESES)}.xlsx"

# ============================================================
# TASAS DE CAMBIO USD
# ============================================================

def limpiar_numero(valor):
    """Convierte valor con formato chileno a float."""
    if isinstance(valor, (int, float)):
        return float(valor)
    valor_str = str(valor).strip()
    return float(valor_str.replace(".", "").replace(",", "."))

USD_RATES = {
    2025: limpiar_numero("961,96"),
    2026: limpiar_numero("897,89"),
}

# ============================================================
# VERIFICACIÓN
# ============================================================

print("\n✓ Librerías importadas")
print(f"\n📁 Carpeta origen: {FOLDER_INPUT}")
print(f"   ¿Existe? {FOLDER_INPUT.exists()}")
print(f"\n📁 Carpeta destino: {FOLDER_OUTPUT}")
print(f"   Archivo salida: {OUTPUT_FILENAME}")
print(f"\n💵 Tasa USD 2025: CLP ${USD_RATES[2025]:,.0f}")
print(f"💵 Tasa USD 2026: CLP ${USD_RATES[2026]:,.0f}")
print(f"\n📆 Reporte configurado: CY={CY} vs LY={LY} | Meses={MESES}")

📂 Rutas configuradas

Origen ventas: C:\Users\Jaime Valderrama\OneDrive - American Sportswear, S.A\Documentos\Jaime\Ventas (Total Año)\BI
Salida reportes: C:\Users\Jaime Valderrama\OneDrive - American Sportswear, S.A\Documentos\Jaime\Ventas (Total Año)\Reportes_Consolidados
Processed: C:\Users\Jaime Valderrama\OneDrive - American Sportswear, S.A\Documentos\Jaime\miprimerproyectods\data\processed
Maestros: C:\Users\Jaime Valderrama\OneDrive - American Sportswear, S.A\Documentos\Jaime\Ventas (Total Año)\BI\Maestros

✓ Librerías importadas

📁 Carpeta origen: C:\Users\Jaime Valderrama\OneDrive - American Sportswear, S.A\Documentos\Jaime\Ventas (Total Año)\BI
   ¿Existe? True

📁 Carpeta destino: C:\Users\Jaime Valderrama\OneDrive - American Sportswear, S.A\Documentos\Jaime\Ventas (Total Año)\Reportes_Consolidados
   Archivo salida: Ventas_Consolidadox_2026_vs_2025_meses_04.xlsx

💵 Tasa USD 2025: CLP $962
💵 Tasa USD 2026: CLP $898

📆 Reporte configurado: CY=2026 vs LY=2025 | Meses=[4]


In [2]:
# ============================================================
# CELDA 2: FUNCIONES
# ============================================================

MAP_CENTRO_COSTO = {
    "E-COMMERCE": "Internet",
    "PARIS": "Internet",
    "MERCADO LIBRE": "Internet",
    "FALABELLA": "Internet",
    "RIPLEY": "Internet",
    "DAFITI": "Internet",
    "CK - COSTANERA CENTER": "Full Price",
    "CK - PARQUE ARAUCO": "Full Price",
    "CK - ALTO LAS CONDES": "Full Price",
    "CK - MARINA ARAUCO": "Full Price",
    "CK - FLORIDA CENTER": "Full Price",
    "CK - VESPUCIO": "Full Price",
    "CK - VIVO IMPERIO": "Full Price",
    "CK - ANTOFAGASTA": "Full Price",
    "CK - PUERTO MONTT": "Full Price",
    "CK - TREBOL": "Full Price",
    "CK - LA SERENA": "Full Price",
    "CK - BUENAVENTURA OUTLET": "Outlet",
    "CK - EASTON OUTLET": "Outlet",
    "CK - VIVO MAIPU OUTLET": "Outlet",
    "CK - CURAUMA OUTLET": "Outlet",
    "CK - COQUIMBO OUTLET": "Outlet",
    "CK - VIVO TEMUCO OUTLET": "Outlet",
    "CK - CHILLAN": "Outlet",
    "CK -PLAZA BIO-BIO": "Outlet",
}

def extraer_anio_desde_nombre(nombre_archivo):
    match = re.search(r"(20\d{2})", nombre_archivo)
    return int(match.group(1)) if match else None

def buscar_archivos_txt(carpeta, years):
    files = [f for f in carpeta.rglob("*.txt") if f.is_file()]
    encontrados = []

    for f in files:
        anio = extraer_anio_desde_nombre(f.name)
        if anio in years:
            encontrados.append({"path": f, "year": anio})

    if not encontrados:
        raise FileNotFoundError(f"No se encontraron TXT para los años {years} en {carpeta}")

    return sorted(encontrados, key=lambda x: (x["year"], x["path"].name))

def leer_txt(path):
    return pd.read_csv(
        path,
        sep="\t",
        encoding="utf-16",
        low_memory=False,
        dtype={
            "Cantidad": str,
            "Precio Unitario": str,
            "Precio tras el descuento": str,
            "Total líneas": str,
            "Costo del artículo": str,
        },
    )

def limpiar_df(df):
    df = df.copy()
    df = df.loc[:, ~df.columns.str.contains("^Unnamed", na=False)]

    df["Marca"] = df["Marca"].fillna("").astype(str).str.strip()
    df = df[df["Marca"].ne("") & df["Marca"].str.lower().ne("nan")].copy()

    df["Fecha Documento"] = pd.to_datetime(df["Fecha Documento"], format="%d/%m/%Y", errors="coerce")

    cols_num = ["Cantidad", "Precio Unitario", "Precio tras el descuento", "Total líneas", "Costo del artículo"]
    for col in cols_num:
        if col in df.columns:
            df[col] = (
                df[col].astype(str)
                .str.replace(".", "", regex=False)
                .str.replace(",", ".", regex=False)
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

def transformar_ventas(df, usd_rate):
    df = df.copy()

    df["Año"] = df["Fecha Documento"].dt.year
    df["Mes Num"] = df["Fecha Documento"].dt.month

    meses_es = {
        1: "enero", 2: "febrero", 3: "marzo", 4: "abril",
        5: "mayo", 6: "junio", 7: "julio", 8: "agosto",
        9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre",
    }

    df["Month"] = df["Mes Num"].apply(lambda x: calendar.month_name[int(x)].capitalize() if pd.notna(x) else None)
    df["Mes"] = df["Mes Num"].map(meses_es)
    df["Country"] = "CHILE"

    df["Marca act"] = df["Marca"]
    df.loc[df["Marca"].str.upper() == "CK LICENSE", "Marca act"] = "CK ACCESSORIES"
    df.loc[df["Marca"].str.upper() == "CK OTHERS", "Marca act"] = "CK Other"

    df["Categ Act"] = df["Categoría"]
    df.loc[df["Categoría"] == "M63-Women-Other Accessories", "Categ Act"] = "M55-Women-OtherAccessories"
    df["Categ Act"] = df["Categ Act"].fillna("Sin Categoria")

    df["Almacén"] = df["Almacén"].fillna("").astype(str).str.strip()
    df["Centro Costo"] = df["Centro Costo"].fillna("").astype(str).str.strip()

    df.loc[df["Almacén"].str.upper() == "MERCADOLIBRE FULLFILLMENT", "Almacén"] = "MERCADO LIBRE"

    marketplaces = ["PARIS", "MERCADO LIBRE", "FALABELLA", "RIPLEY", "DAFITI"]
    mask_cambiar = (
        (df["Almacén"].str.upper() == "CK - ECOMMERCE") &
        (df["Centro Costo"].str.upper().isin(marketplaces))
    )
    df.loc[mask_cambiar, "Almacén"] = df.loc[mask_cambiar, "Centro Costo"]

    df["Nombre de tipo de centro de costo"] = df["Centro Costo"].map(MAP_CENTRO_COSTO).fillna("Sin clasificar")

    df["Combi"] = df["Número de artículo"]
    df["Genero"] = np.select(
        [
            df["Categ Act"].astype(str).str.contains("Women", case=False, na=False),
            df["Categ Act"].astype(str).str.contains("Men", case=False, na=False),
            df["Categ Act"].astype(str).str.contains("Kids", case=False, na=False),
        ],
        ["Women", "Men", "Kids"],
        default="Other"
    )

    df["Vta Neta"] = df["Total líneas"]
    df["Unidades"] = df["Cantidad"]
    df["Vta USD"] = df["Vta Neta"] / usd_rate

    df["Fecha Documento"] = df["Fecha Documento"].dt.strftime("%d/%m/%Y")
    return df

In [3]:
# ============================================================
# CELDA 3: PROCESAMIENTO AUTOMÁTICO
# ============================================================

archivos = buscar_archivos_txt(FOLDER_INPUT, years=[LY, CY])

print("📂 ARCHIVOS DETECTADOS:")
for a in archivos:
    print(f"   {a['year']}: {a['path'].name}")

dfs_transformados = []

for item in archivos:
    year = item["year"]
    path = item["path"]

    df_raw = leer_txt(path)
    df_clean = limpiar_df(df_raw)

    df_clean = df_clean[df_clean["Fecha Documento"].dt.month.isin(MESES)].copy()

    usd_rate = USD_RATES.get(year)
    if usd_rate is None:
        raise ValueError(f"No hay tasa USD para el año {year}")

    df_trans = transformar_ventas(df_clean, usd_rate)
    dfs_transformados.append((year, df_trans))

    print(f"\n✓ Año {year}: {path.name}")
    print(f"   Filas: {len(df_trans):,}")
    print(f"   Venta neta: ${df_trans['Vta Neta'].sum():,.0f}")

df_2025 = [df for year, df in dfs_transformados if year == LY][0]
df_2026 = [df for year, df in dfs_transformados if year == CY][0]

group_cols = [
    "Country", "Fecha Documento", "Mes", "Month", "Año",
    "Nombre de tipo de centro de costo", "Almacén",
    "Marca", "Marca act", "Categoría", "Categ Act",
    "Combi", "Descripción", "Genero",
]

df_2025_grouped = (
    df_2025.groupby(group_cols, as_index=False, dropna=False)
    .agg({"Vta Neta": "sum", "Unidades": "sum", "Vta USD": "sum"})
    .rename(columns={"Vta Neta": "Vta Neta LY", "Unidades": "Uni LY", "Vta USD": "Vta U$ LY"})
)

df_2026_grouped = (
    df_2026.groupby(group_cols, as_index=False, dropna=False)
    .agg({"Vta Neta": "sum", "Unidades": "sum", "Vta USD": "sum"})
    .rename(columns={"Vta Neta": "Vta Neta CY", "Unidades": "Uni CY", "Vta USD": "Vta U$ CY"})
)

print(f"\nCheck {LY}: {df_2025['Vta Neta'].sum():,.0f} vs agrupado: {df_2025_grouped['Vta Neta LY'].sum():,.0f}")
print(f"Check {CY}: {df_2026['Vta Neta'].sum():,.0f} vs agrupado: {df_2026_grouped['Vta Neta CY'].sum():,.0f}")

df_final = pd.merge(df_2026_grouped, df_2025_grouped, on=group_cols, how="outer")

for c in ["Vta Neta LY", "Vta Neta CY", "Uni LY", "Uni CY", "Vta U$ LY", "Vta U$ CY"]:
    df_final[c] = df_final[c].fillna(0)

df_final = df_final[
    [
        "Country", "Fecha Documento", "Mes", "Month", "Año",
        "Nombre de tipo de centro de costo", "Almacén",
        "Marca", "Marca act", "Categoría", "Categ Act",
        "Combi", "Descripción", "Genero",
        "Vta Neta LY", "Vta Neta CY",
        "Uni LY", "Uni CY",
        "Vta U$ LY", "Vta U$ CY",
    ]
].sort_values(
    ["Country", "Fecha Documento", "Nombre de tipo de centro de costo", "Almacén", "Marca", "Marca act", "Categoría", "Categ Act", "Combi"],
    ascending=True
).reset_index(drop=True)

print("\n" + "=" * 60)
print("📈 RESUMEN FINAL")
print("=" * 60)
print(f"\nVentas LY: ${df_final['Vta Neta LY'].sum():,.0f} CLP ({df_final['Uni LY'].sum():,.0f} unidades)")
print(f"Ventas CY: ${df_final['Vta Neta CY'].sum():,.0f} CLP ({df_final['Uni CY'].sum():,.0f} unidades)")
print(f"\nVentas USD LY: ${df_final['Vta U$ LY'].sum():,.2f}")
print(f"Ventas USD CY: ${df_final['Vta U$ CY'].sum():,.2f}")
print(f"\nProductos únicos (Combi): {df_final['Combi'].nunique():,}")
print(f"Productos solo LY: {(df_final['Vta Neta CY'] == 0).sum():,}")
print(f"Productos solo CY: {(df_final['Vta Neta LY'] == 0).sum():,}")
print(f"Productos ambos años: {((df_final['Vta Neta LY'] > 0) & (df_final['Vta Neta CY'] > 0)).sum():,}")

display(df_final.head(10))

output_path = FOLDER_OUTPUT / OUTPUT_FILENAME
print("\n💾 EXPORTANDO A EXCEL...")
df_final.to_excel(output_path, index=False, engine="openpyxl")

print("\n✅ PROCESO COMPLETADO")
print(f"📁 Archivo guardado en: {output_path}")
print(f"📊 Tamaño: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

📂 ARCHIVOS DETECTADOS:
   2025: Informe Ventas 2025.txt
   2026: Informe Ventas 2026.txt

✓ Año 2025: Informe Ventas 2025.txt
   Filas: 49,833
   Venta neta: $1,412,485,912

✓ Año 2026: Informe Ventas 2026.txt
   Filas: 42,607
   Venta neta: $1,464,186,301

Check 2025: 1,412,485,912 vs agrupado: 1,412,485,912
Check 2026: 1,464,186,301 vs agrupado: 1,464,186,301

📈 RESUMEN FINAL

Ventas LY: $1,412,485,912 CLP (52,239 unidades)
Ventas CY: $1,464,186,301 CLP (54,424 unidades)

Ventas USD LY: $1,468,341.63
Ventas USD CY: $1,630,696.75

Productos únicos (Combi): 19,362
Productos solo LY: 43,572
Productos solo CY: 39,037
Productos ambos años: 0


,Country,Fecha Documento,Mes,Month,Año,Nombre de tipo de centro de costo,Almacén,Marca,Marca act,Categoría,Categ Act,Combi,Descripción,Genero,Vta Neta LY,Vta Neta CY,Uni LY,Uni CY,Vta U$ LY,Vta U$ CY
0,CHILE,01/04/2025,abril,April,2025,Full Price,CK - Alto Las Condes,CK ACCESSORIES,CK ACCESSORIES,M55-Women-Bags,M55-Women-Bags,K60K6125660GMOS,CK BUSINESS CAMERA BAG_MONO,Women,73101.00,0.00,1.00,0.00,75.99,0.00
1,CHILE,01/04/2025,abril,April,2025,Full Price,CK - Alto Las Condes,CK JEANS,CK JEANS,M53-Men-Heavyweight Knits,M53-Men-Heavyweight Knits,J30J325630L5YM,MONOLOGO CREW NECK,Men,67219.00,0.00,1.00,0.00,69.88,0.00
2,CHILE,01/04/2025,abril,April,2025,Full Price,CK - Alto Las Condes,CK JEANS,CK JEANS,M53-Men-Polos S/S,M53-Men-Polos S/S,J30J323394CFFXL,BADGE POLO,Men,-36968.00,0.00,-1.00,0.00,-38.43,0.00
3,CHILE,01/04/2025,abril,April,2025,Full Price,CK - Alto Las Condes,CK JEANS,CK JEANS,M53-Men-Polos S/S,M53-Men-Polos S/S,J30J323394CFFXXL,BADGE POLO,Men,36968.00,0.00,1.00,0.00,38.43,0.00
4,CHILE,01/04/2025,abril,April,2025,Full Price,CK - Alto Las Condes,CK JEANS,CK JEANS,M53-Men-Polos S/S,M53-Men-Polos S/S,J30J325269CHWXL,CK EMBRO BADGE SLIM POLO,Men,-36968.00,0.00,-1.00,0.00,-38.43,0.00
5,CHILE,01/04/2025,abril,April,2025,Full Price,CK - Alto Las Condes,CK JEANS,CK JEANS,M53-Men-T-Shirts S/S,M53-Men-T-Shirts S/S,J30J325916BEHXXL,MONOLOGO APPLIQUE TEE,Men,33605.00,0.00,1.00,0.00,34.93,0.00
6,CHILE,01/04/2025,abril,April,2025,Full Price,CK - Alto Las Condes,CK JEANS,CK JEANS,M53-Women-Dresses,M53-Women-Dresses,J20J2237121A4M,SLEEVELESS DENIM DRESS,Women,73101.00,0.00,1.00,0.00,75.99,0.00
7,CHILE,01/04/2025,abril,April,2025,Full Price,CK - Alto Las Condes,CK JEANS,CK JEANS,M53-Women-Heavyweight Knits,M53-Women-Heavyweight Knits,J20J222539AATS,WOVEN LABEL OVERSIZED HOODIE,Women,84026.00,0.00,1.00,0.00,87.35,0.00
8,CHILE,01/04/2025,abril,April,2025,Full Price,CK - Alto Las Condes,CK JEANS,CK JEANS,M53-Women-Heavyweight Knits,M53-Women-Heavyweight Knits,J20J223078BEHXS,LOGO ELASTIC HOODIE,Women,75622.00,0.00,1.00,0.00,78.61,0.00
9,CHILE,01/04/2025,abril,April,2025,Full Price,CK - Alto Las Condes,CK JEANS,CK JEANS,M53-Women-T-Shirts S/S,M53-Women-T-Shirts S/S,J20J223166BEHXS,WARP LOGO BOYFRIEND TEE,Women,39487.00,0.00,1.00,0.00,41.05,0.00



💾 EXPORTANDO A EXCEL...

✅ PROCESO COMPLETADO
📁 Archivo guardado en: C:\Users\Jaime Valderrama\OneDrive - American Sportswear, S.A\Documentos\Jaime\Ventas (Total Año)\Reportes_Consolidados\Ventas_Consolidadox_2026_vs_2025_meses_04.xlsx
📊 Tamaño: 7.32 MB
